# Student Performance Dashboard

This notebook loads student performance data, runs SQL queries via SQLite, and visualises the results as a 2×2 dashboard.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os

# ------------------------------------------------------------------
# Load CSV
# ------------------------------------------------------------------
file_path = 'student_performance.csv'

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Cannot proceed without '{file_path}'. "
        "Please place the CSV in the same directory as this notebook."
    )

df = pd.read_csv(file_path)
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# ------------------------------------------------------------------
# Persist to in-memory SQLite and run analytical queries
# ------------------------------------------------------------------
conn = sqlite3.connect(":memory:")
df.to_sql('students', conn, if_exists='replace', index=False)

# 1. Average math score per department
avg_math_df = pd.read_sql_query("""
    SELECT department,
           ROUND(AVG(math_score), 1) AS avg_math_score
    FROM students
    GROUP BY department
    ORDER BY avg_math_score DESC
""", conn)

# 2. Student count per department
dept_count_df = pd.read_sql_query("""
    SELECT department, COUNT(*) AS student_count
    FROM students
    GROUP BY department
""", conn)

# 3. Top 8 students by total score
top_students_df = pd.read_sql_query("""
    SELECT name,
           (math_score + science_score + english_score + programming_score) AS total_score
    FROM students
    ORDER BY total_score DESC
    LIMIT 8
""", conn)

# 4. Average attendance by gender
attendance_df = pd.read_sql_query("""
    SELECT gender,
           ROUND(AVG(attendance_percentage), 1) AS avg_attendance
    FROM students
    GROUP BY gender
""", conn)

conn.close()

print("Average Math Score by Department")
print(avg_math_df)
print("\nStudent Count by Department")
print(dept_count_df)
print("\nTop 8 Students")
print(top_students_df)
print("\nAverage Attendance by Gender")
print(attendance_df)

In [ ]:
# ------------------------------------------------------------------
# Build 2×2 Dashboard
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1 – Vertical bar: avg math score by department
ax1 = axes[0, 0]
bars1 = ax1.bar(avg_math_df["department"], avg_math_df["avg_math_score"],
                color="skyblue", edgecolor="black")
ax1.set_title("Average Math Score by Department", fontsize=13)
ax1.set_xlabel("Department")
ax1.set_ylabel("Average Math Score")
ax1.set_ylim(0, 105)
ax1.grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f"{bar.get_height()}", ha='center', fontsize=10)

# Panel 2 – Pie: student distribution by department
ax2 = axes[0, 1]
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0']
ax2.pie(dept_count_df["student_count"], labels=dept_count_df["department"],
        autopct='%1.1f%%', startangle=140, colors=colors,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title("Student Distribution by Department", fontsize=13)

# Panel 3 – Horizontal bar: top 8 students
ax3 = axes[1, 0]
top_plot = top_students_df.iloc[::-1]
bars3 = ax3.barh(top_plot["name"], top_plot["total_score"],
                 color="mediumseagreen", edgecolor="black")
ax3.set_title("Top 8 Students by Total Score", fontsize=13)
ax3.set_xlabel("Total Score")
ax3.grid(axis='x', linestyle='--', alpha=0.6)
for bar in bars3:
    ax3.text(bar.get_width() - 15, bar.get_y() + bar.get_height() / 2,
             f"{int(bar.get_width())}", va='center',
             color='white', fontsize=9, fontweight='bold')

# Panel 4 – Bar: avg attendance by gender
ax4 = axes[1, 1]
bars4 = ax4.bar(attendance_df["gender"], attendance_df["avg_attendance"],
                color=['#ff7f50', '#6495ed'], edgecolor='black')
ax4.set_title("Average Attendance Percentage by Gender", fontsize=13)
ax4.set_xlabel("Gender")
ax4.set_ylabel("Attendance %")
ax4.set_ylim(0, 105)
ax4.grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars4:
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f"{bar.get_height()}%", ha='center', fontsize=10)

fig.suptitle("Student Performance Dashboard", fontsize=20, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("student_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print("\nDashboard saved as 'student_dashboard.png'")